# Project 1: Parts 1 and 2

## 1. Dataset selection

Two public datasets in the residential property market domain were selected: **House Sales in King County, USA** (Kaggle publisher: harlfoxem) and **Melbourne Housing Market** (Kaggle publisher: Anthony Pino). Both contain property characteristics, location information, recorded dates and a numeric price variable. These common themes support later comparison of completeness, consistency and usefulness.

- King County source: https://www.kaggle.com/datasets/harlfoxem/housesalesprediction
- Melbourne source: https://www.kaggle.com/datasets/anthonypino/melbourne-housing-market
- Sources and data accessed: 2 November 2026. Downloaded publisher descriptions are retained alongside this report.

Use `kc_house_data.csv` and **`Melbourne_housing_FULL.csv`**. The Melbourne download also contains `MELBOURNE_HOUSE_PRICES_LESS.csv`, but that file has only 13 columns and is unsuitable for the stated feature requirement.

Both selected files have 21 original columns: 20 non-target columns plus one target. King County still has 19 candidate predictors after excluding its property identifier; Melbourne still has 19 after excluding Address. These counts exceed 15. Identifiers and date fields are described explicitly rather than silently counted as continuous measurements.

The datasets represent different cities, years, currencies and property mixes. Compare structure and data quality separately; raw price differences cannot establish which city is more expensive. Melbourne includes unsuccessful or undisclosed sale outcomes as well as sales, so not every row represents a completed sale with an observed price. No rows are removed, imputed or altered in Parts 1 and 2.


## 2. Data loading and overview

Run all cells in order. Dependencies: Python, pandas and a Jupyter environment. The CSVs are supplied locally; no account or network connection is needed to run this notebook.

In [1]:
from pathlib import Path
import pandas as pd

# Run from this folder, or from the original Data Science workspace.
BASE = Path.cwd()
if not (BASE / "data" / "kc_house_data.csv").exists():
    BASE = BASE / "outputs" / "project1_parts1_2"
if not (BASE / "data" / "kc_house_data.csv").exists():
    raise FileNotFoundError("Open the notebook in the supplied project1_parts1_2 folder, keeping its data folder beside it.")

king_county = pd.read_csv(BASE / "data" / "kc_house_data.csv")
melbourne = pd.read_csv(BASE / "data" / "Melbourne_housing_FULL.csv")
datasets = {"King County": king_county, "Melbourne": melbourne}
targets = {"King County": "price", "Melbourne": "Price"}


In [2]:
overview_rows = []
for name, df in datasets.items():
    date_column = "date" if name == "King County" else "Date"
    dates = (pd.to_datetime(df[date_column], format="%Y%m%dT%H%M%S", errors="raise")
             if name == "King County" else
             pd.to_datetime(df[date_column], format="%d/%m/%Y", errors="raise"))
    assert df.shape[0] >= 3000 and df.shape[1] - 2 >= 15
    overview_rows.append({
        "Dataset": name, "Rows": len(df), "Columns": df.shape[1],
        "Non-target columns": df.shape[1] - 1,
        "Start date": dates.min().date().isoformat(),
        "End date": dates.max().date().isoformat(),
        "Target": targets[name],
        "Currency": "USD" if name == "King County" else "AUD",
        "Observed target": int(df[targets[name]].notna().sum()),
        "Missing target": int(df[targets[name]].isna().sum()),
    })
overview = pd.DataFrame(overview_rows)
display(overview)


,Dataset,Rows,Columns,Non-target columns,Start date,End date,Target,Currency,Observed target,Missing target
0,King County,21613,21,20,2014-05-02,2015-05-27,price,USD,21613,0
1,Melbourne,34857,21,20,2016-01-28,2018-03-17,Price,AUD,27247,7610


In [3]:
for name, df in datasets.items():
    print(f"\n{name}: first five rows")
    display(df.head())
    print("\nPandas structure:")
    df.info()


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,68 Studley St,2,h,NaN,SS,Jellis,3/09/2016,2.5,3067.0,2.0,1.0,1.0,126.0,NaN,NaN,Yarra City Council,-37.8014,144.9958,Northern Metropolitan,4019.0
1,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,2.0,1.0,1.0,202.0,NaN,NaN,Yarra City Council,-37.7996,144.9984,Northern Metropolitan,4019.0
2,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,2.0,1.0,0.0,156.0,79.0,1900.0,Yarra City Council,-37.8079,144.9934,Northern Metropolitan,4019.0
3,Abbotsford,18/659 Victoria St,3,u,NaN,VB,Rounds,4/02/2016,2.5,3067.0,3.0,2.0,1.0,0.0,NaN,NaN,Yarra City Council,-37.8114,145.0116,Northern Metropolitan,4019.0
4,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,3.0,2.0,0.0,134.0,150.0,1900.0,Yarra City Council,-37.8093,144.9944,Northern Metropolitan,4019.0



King County: first five rows

Pandas structure:
<class 'pandas.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  str    
 2   price          21613 non-null  float64
 3   bedrooms       21613 non-null  int64  
 4   bathrooms      21613 non-null  float64
 5   sqft_living    21613 non-null  int64  
 6   sqft_lot       21613 non-null  int64  
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  int64  
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  int64  
 11  grade          21613 non-null  int64  
 12  sqft_above     21613 non-null  int64  
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat         

In [4]:
definitions = {'King County': [('id', 'Identifier', 'None', 'Property identifier; the same property can appear in multiple sales.'), ('date', 'Date/time', 'Calendar date', 'Sale date stored as YYYYMMDDTHHMMSS text.'), ('price', 'Continuous target', 'USD', 'Recorded house sale price.'), ('bedrooms', 'Discrete count', 'Bedrooms', 'Number of bedrooms.'), ('bathrooms', 'Numeric count', 'Bathroom equivalents', 'Bathroom count; fractional values represent partial bathrooms.'), ('sqft_living', 'Continuous area', 'Square feet', 'Interior living area.'), ('sqft_lot', 'Continuous area', 'Square feet', 'Land lot area.'), ('floors', 'Numeric count', 'Floors', 'Number of floors or levels, including fractional levels.'), ('waterfront', 'Binary category', '0/1', 'Indicator for a waterfront property.'), ('view', 'Ordinal category', 'Rating code', 'View rating; coded values are categories, not physical units.'), ('condition', 'Ordinal category', 'Rating code', 'Overall property condition rating.'), ('grade', 'Ordinal category', 'Rating code', 'Construction/design grade under the King County grading system.'), ('sqft_above', 'Continuous area', 'Square feet', 'Interior area above ground, excluding basement.'), ('sqft_basement', 'Continuous area', 'Square feet', 'Basement area; zero can indicate no basement.'), ('yr_built', 'Calendar year', 'Year', 'Year the property was built.'), ('yr_renovated', 'Calendar year / code', 'Year; 0 sentinel', 'Renovation year; zero indicates no renovation recorded, not year zero.'), ('zipcode', 'Nominal category', 'Postal code', 'ZIP code; numeric storage does not make this a quantitative measurement.'), ('lat', 'Continuous coordinate', 'Decimal degrees', 'Latitude.'), ('long', 'Continuous coordinate', 'Decimal degrees', 'Longitude.'), ('sqft_living15', 'Continuous area', 'Square feet', 'Neighborhood living-area comparison field. Exact aggregation is not specified in the downloaded publisher description; verify before substantive interpretation.'), ('sqft_lot15', 'Continuous area', 'Square feet', 'Neighborhood lot-area comparison field. Exact aggregation is not specified in the downloaded publisher description; verify before substantive interpretation.')], 'Melbourne': [('Suburb', 'Nominal category', 'None', 'Suburb name.'), ('Address', 'Identifier / text', 'None', 'Property street address; not guaranteed to be unique across listings.'), ('Rooms', 'Discrete count', 'Rooms', 'Number of rooms, using the publisher wording.'), ('Type', 'Nominal category', 'Code', 'h = house/cottage/villa/semi/terrace; u = unit/duplex; t = townhouse.'), ('Price', 'Continuous target', 'AUD', 'Recorded property price in Australian dollars; may be missing.'), ('Method', 'Nominal category', 'Code', 'Sale/auction outcome code; see the code legend below.'), ('SellerG', 'Nominal category', 'None', 'Selling real estate agent or agency.'), ('Date', 'Date/time', 'Calendar date', 'Recorded sale/listing event date; stored in day/month/year format.'), ('Distance', 'Continuous distance', 'Kilometres', 'Distance from Melbourne central business district (CBD).'), ('Postcode', 'Nominal category', 'Postal code', 'Postal code; numeric storage does not imply a continuous measurement.'), ('Bedroom2', 'Discrete count', 'Bedrooms', 'Bedroom count scraped from a different source; do not automatically equate it with Rooms.'), ('Bathroom', 'Discrete count', 'Bathrooms', 'Number of bathrooms.'), ('Car', 'Discrete count', 'Car spaces', 'Number of parking spaces.'), ('Landsize', 'Continuous area', 'Square metres (interpreted)', 'Land area. Publisher says "Metres"; square metres is the area interpretation and should be verified before unit conversion.'), ('BuildingArea', 'Continuous area', 'Square metres (interpreted)', 'Building area. Publisher says "Metres"; square metres is the area interpretation and should be verified before unit conversion.'), ('YearBuilt', 'Calendar year', 'Year', 'Year the property was built.'), ('CouncilArea', 'Nominal category', 'None', 'Governing local council.'), ('Lattitude', 'Continuous coordinate', 'Decimal degrees', 'Latitude; original column spelling is preserved.'), ('Longtitude', 'Continuous coordinate', 'Decimal degrees', 'Longitude; original column spelling is preserved.'), ('Regionname', 'Nominal category', 'None', 'General geographical region.'), ('Propertycount', 'Discrete count', 'Properties', 'Number of properties in the suburb.')]}

feature_dictionaries = {}
for name, df in datasets.items():
    dictionary = pd.DataFrame(definitions[name], columns=["Feature", "Meaning/type", "Unit or coding", "Description"])
    assert dictionary["Feature"].tolist() == df.columns.tolist()
    dictionary.insert(1, "Pandas dtype", [str(df[c].dtype) for c in dictionary["Feature"]])
    dictionary["Missing count"] = [int(df[c].isna().sum()) for c in dictionary["Feature"]]
    feature_dictionaries[name] = dictionary
    print(f"\n{name}: feature descriptions")
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(dictionary)


,Feature,Pandas dtype,Meaning/type,Unit or coding,Description,Missing count
0,id,int64,Identifier,None,Property identifier; the same property can appear in multiple sales.,0
1,date,str,Date/time,Calendar date,Sale date stored as YYYYMMDDTHHMMSS text.,0
2,price,float64,Continuous target,USD,Recorded house sale price.,0
3,bedrooms,int64,Discrete count,Bedrooms,Number of bedrooms.,0
4,bathrooms,float64,Numeric count,Bathroom equivalents,Bathroom count; fractional values represent partial bathrooms.,0
5,sqft_living,int64,Continuous area,Square feet,Interior living area.,0
6,sqft_lot,int64,Continuous area,Square feet,Land lot area.,0
7,floors,float64,Numeric count,Floors,"Number of floors or levels, including fractional levels.",0
8,waterfront,int64,Binary category,0/1,Indicator for a waterfront property.,0
9,view,int64,Ordinal category,Rating code,"View rating; coded values are categories, not physical units.",0


,Feature,Pandas dtype,Meaning/type,Unit or coding,Description,Missing count
0,Suburb,str,Nominal category,None,Suburb name.,0
1,Address,str,Identifier / text,None,Property street address; not guaranteed to be unique across listings.,0
2,Rooms,int64,Discrete count,Rooms,"Number of rooms, using the publisher wording.",0
3,Type,str,Nominal category,Code,h = house/cottage/villa/semi/terrace; u = unit/duplex; t = townhouse.,0
4,Price,float64,Continuous target,AUD,Recorded property price in Australian dollars; may be missing.,7610
5,Method,str,Nominal category,Code,Sale/auction outcome code; see the code legend below.,0
6,SellerG,str,Nominal category,None,Selling real estate agent or agency.,0
7,Date,str,Date/time,Calendar date,Recorded sale/listing event date; stored in day/month/year format.,0
8,Distance,float64,Continuous distance,Kilometres,Distance from Melbourne central business district (CBD).,1
9,Postcode,float64,Nominal category,Postal code,Postal code; numeric storage does not imply a continuous measurement.,1



King County: feature descriptions

Melbourne: feature descriptions


## 2. Data loading and overview: interpretation

`pandas.read_csv()` loads each original CSV into a separate DataFrame. Date parsing is performed only in temporary Series to verify date coverage; the raw DataFrames retain their original columns and values. The tables below report observed pandas dtypes, semantic types, units/codes and basic missing counts. Dtypes may show `str` or `object` for text depending on the installed pandas version. Counts stored as floats can reflect missing entries rather than fractional counts.

**Target variables:** King County uses `price` in USD; Melbourne uses `Price` in AUD. Both are numeric continuous outcomes suitable for a possible regression task. Predictive modelling is not required for Parts 1 and 2. Missing Melbourne prices remain missing; a later modelling task must decide how to handle them and distinguish non-sale outcomes from undisclosed prices.

**Melbourne Method legend (publisher definitions):** S = sold; SP = sold prior; PI = passed in; PN = sold prior, price undisclosed; SN = sold, price undisclosed; NB = no bid; VB = vendor bid; W = withdrawn prior to auction; SA = sold after auction; SS = sold after auction, price undisclosed. Some publisher-defined codes may not occur in this file. These codes should remain categorical.

The absence of pandas-detected nulls in King County does not establish that it has perfect quality. Sentinel codes, repeated property sales and implausible values need assessment in later parts. Parts 3-6, including visual exploration and a final quality ranking, remain for the rest of the group.


In [5]:
# Save tables for the group report without modifying the original data.
overview.to_csv(BASE / "dataset_overview.csv", index=False)
for name, dictionary in feature_dictionaries.items():
    dictionary.to_csv(BASE / (name.lower().replace(" ", "_") + "_feature_dictionary.csv"), index=False)
print("Saved overview and both feature dictionaries.")


Saved overview and both feature dictionaries.
